# Knowledge-Graph-Enhanced Disease Prediction (v2 — final)
**Pipeline:** symptom dataset + Hetionet knowledge graph → 6 model variants compared under missing symptoms → explainability → data-quality layer → auto-generated README.

**Run:** zip file `/content` mein upload karo → *Runtime → Run all*. Aakhir mein `results.zip` download hogi (README, plots, CSVs sab andar).

## 1. Zip extract

In [ ]:
import os, glob, zipfile, re
import numpy as np, pandas as pd

BASE = '/content'
DATA = f'{BASE}/data'; KG_DIR = f'{BASE}/kg'; OUT = f'{BASE}/results'
for d in (DATA, KG_DIR, OUT): os.makedirs(d, exist_ok=True)

zips = glob.glob(f'{BASE}/*.zip')
print('Zip files mili:', zips)
for z in zips:
    with zipfile.ZipFile(z) as f:
        f.extractall(DATA)

for root, _, files in os.walk(DATA):
    for fn in files:
        print(os.path.join(root, fn))

## 2. Dataset load + clean
Dono formats support hain: `Training.csv`/`Testing.csv` (one-hot + `prognosis`) aur `dataset.csv` (Symptom_1..17). Duplicates hata diye jate hain taake train/test leakage na ho.

In [ ]:
csvs = glob.glob(f'{DATA}/**/*.csv', recursive=True)
print('CSV files:', csvs)

def norm(s):
    return re.sub(r'[\s_]+', '_', str(s).strip().lower()).strip('_')

records = []
for p in csvs:
    f = pd.read_csv(p)
    f = f.loc[:, ~f.columns.str.startswith('Unnamed')]
    if 'prognosis' in f.columns:
        sym_cols = [c for c in f.columns if c != 'prognosis']
        for _, r in f.iterrows():
            syms = tuple(sorted({norm(re.sub(r'\.\d+$', '', c)) for c in sym_cols if r[c] == 1}))
            records.append((' '.join(str(r['prognosis']).split()), syms))
    elif 'Disease' in f.columns:
        sym_cols = [c for c in f.columns if c.lower().startswith('symptom')]
        for _, r in f.iterrows():
            syms = tuple(sorted({norm(r[c]) for c in sym_cols if pd.notna(r[c]) and str(r[c]).strip()}))
            records.append((' '.join(str(r['Disease']).split()), syms))
    else:
        print('Skipped (format nahi pehchana):', p)

raw_rows = len(records)
df = pd.DataFrame(records, columns=['disease', 'symptoms'])
df = df[df.symptoms.map(len) > 0].drop_duplicates().reset_index(drop=True)
print(f'Raw rows: {raw_rows} | Unique records: {len(df)} | Diseases: {df.disease.nunique()}')

all_syms = sorted({s for ss in df.symptoms for s in ss})
sym_idx = {s: i for i, s in enumerate(all_syms)}
print('Unique symptoms:', len(all_syms))

def onehot(sym_lists):
    X = np.zeros((len(sym_lists), len(all_syms)))
    for i, ss in enumerate(sym_lists):
        for s in ss:
            if s in sym_idx: X[i, sym_idx[s]] = 1
    return X
df.head()

## 3. Hetionet knowledge graph (Disease–presents–Symptom edges)

In [ ]:
import networkx as nx
URL = 'https://github.com/hetio/hetionet/raw/main/hetnet/tsv/'
for fn in ['hetionet-v1.0-nodes.tsv', 'hetionet-v1.0-edges.sif.gz']:
    path = f'{KG_DIR}/{fn}'
    if not os.path.exists(path) or os.path.getsize(path) < 100_000:
        os.system(f'wget -q -O {path} {URL}{fn}')
    size_kb = os.path.getsize(path) // 1024
    print(f'{fn}: {size_kb} KB')
    assert size_kb > 100, f'{fn} theek download nahi hui — GitHub (hetio/hetionet) se manually download karke {KG_DIR} mein daalo'

nodes = pd.read_csv(f'{KG_DIR}/hetionet-v1.0-nodes.tsv', sep='\t')
edges = pd.read_csv(f'{KG_DIR}/hetionet-v1.0-edges.sif.gz', sep='\t')
id2name = dict(zip(nodes['id'], nodes['name'].str.lower()))
dps = edges[edges['metaedge'] == 'DpS']

G = nx.Graph()
for src, tgt in zip(dps['source'], dps['target']):
    d, s = id2name[src], id2name[tgt]
    G.add_node(d, kind='Disease'); G.add_node(s, kind='Symptom')
    G.add_edge(d, s)

kg_diseases = {n for n, a in G.nodes(data=True) if a['kind'] == 'Disease'}
kg_symptoms = {n for n, a in G.nodes(data=True) if a['kind'] == 'Symptom'}
kg_dis_syms = {d: set(G.neighbors(d)) for d in kg_diseases}
print(f'KG: {len(kg_diseases)} diseases, {len(kg_symptoms)} symptoms, {G.number_of_edges()} DpS edges')

## 4. Dataset ↔ KG mapping

In [ ]:
SYMPTOM_MAP = {
 'itching': ['pruritus'], 'internal_itching': ['pruritus'], 'irritation_in_anus': ['pruritus ani'],
 'skin_rash': ['skin manifestations'], 'nodal_skin_eruptions': ['skin manifestations'],
 'red_spots_over_body': ['purpura', 'skin manifestations'], 'skin_peeling': ['skin manifestations'],
 'dischromic_patches': ['skin manifestations'], 'pus_filled_pimples': ['skin manifestations'],
 'blackheads': ['skin manifestations'], 'blister': ['skin manifestations'],
 'silver_like_dusting': ['skin manifestations'], 'yellow_crust_ooze': ['skin manifestations'],
 'bruising': ['ecchymosis'], 'continuous_sneezing': ['sneezing'],
 'shivering': ['chills'], 'chills': ['chills'], 'high_fever': ['fever'], 'mild_fever': ['fever'],
 'joint_pain': ['arthralgia'], 'knee_pain': ['arthralgia'], 'hip_joint_pain': ['arthralgia'],
 'swelling_joints': ['arthralgia'], 'painful_walking': ['mobility limitation'],
 'movement_stiffness': ['muscle rigidity'], 'stiff_neck': ['meningism', 'neck pain'],
 'muscle_pain': ['myalgia'], 'muscle_weakness': ['muscle weakness'], 'weakness_in_limbs': ['muscle weakness'],
 'muscle_wasting': ['muscular atrophy'], 'cramps': ['muscle cramp'],
 'stomach_pain': ['abdominal pain'], 'abdominal_pain': ['abdominal pain'], 'belly_pain': ['abdominal pain'],
 'acidity': ['heartburn'], 'indigestion': ['dyspepsia'], 'passage_of_gases': ['flatulence'],
 'diarrhoea': ['diarrhea'], 'stomach_bleeding': ['hematemesis'], 'loss_of_appetite': ['anorexia'],
 'excessive_hunger': ['hyperphagia'], 'increased_appetite': ['hyperphagia'],
 'yellowish_skin': ['jaundice'], 'yellowing_of_eyes': ['jaundice'],
 'breathlessness': ['dyspnea'], 'blood_in_sputum': ['hemoptysis'],
 'spinning_movements': ['vertigo'], 'loss_of_balance': ['gait disorders, neurologic'], 'unsteadiness': ['ataxia'],
 'blurred_and_distorted_vision': ['vision disorders'], 'visual_disturbances': ['vision disorders'],
 'pain_behind_the_eyes': ['eye pain'], 'loss_of_smell': ['olfaction disorders'],
 'slurred_speech': ['dysarthria'], 'weakness_of_one_body_side': ['hemiplegia'],
 'altered_sensorium': ['consciousness disorders'], 'lack_of_concentration': ['memory disorders'],
 'restlessness': ['psychomotor agitation'], 'malaise': ['asthenia'],
 'drying_and_tingling_lips': ['paresthesia'], 'ulcers_on_tongue': ['oral manifestations'],
 'swollen_legs': ['edema'], 'puffy_face_and_eyes': ['edema'], 'swollen_extremeties': ['edema'], 'fluid_overload': ['edema'],
 'burning_micturition': ['dysuria'], 'spotting_urination': ['dysuria'],
 'continuous_feel_of_urine': ['lower urinary tract symptoms'], 'bladder_discomfort': ['pelvic pain'],
 'abnormal_menstruation': ['dysmenorrhea'],
}
DISEASE_MAP = {   # (approx) = close but not exact match — README mein mention karna
 'diabetes': ['type 2 diabetes mellitus'], 'bronchial asthma': ['asthma'], 'hypertension': ['hypertension'],
 'migraine': ['migraine'], 'osteoarthristis': ['osteoarthritis'], 'arthritis': ['rheumatoid arthritis'],
 'psoriasis': ['psoriasis'], 'malaria': ['malaria'], 'hepatitis b': ['hepatitis b'],
 'hypothyroidism': ['hypothyroidism'], 'aids': ['acquired immunodeficiency syndrome'],
 'allergy': ['allergic rhinitis'],                     # (approx)
 'hyperthyroidism': ["graves' disease"],               # (approx)
 'heart attack': ['coronary artery disease'],          # (approx)
 'chronic cholestasis': ['primary biliary cirrhosis'], # (approx)
 'cervical spondylosis': ['degenerative disc disease'],# (approx)
}

def resolve(name, manual, kg_set):
    for c in manual.get(name.lower(), []) + [name.lower().replace('_', ' ')]:
        if c in kg_set:
            return c
    return None

sym_map = {s: k for s in all_syms if (k := resolve(s, SYMPTOM_MAP, kg_symptoms))}
dis_map = {d: k for d in sorted(df.disease.unique()) if (k := resolve(d, DISEASE_MAP, kg_diseases))}

print(f'Symptoms mapped: {len(sym_map)}/{len(all_syms)} ({len(sym_map)/len(all_syms):.0%})')
print(f'Diseases mapped: {len(dis_map)}/{df.disease.nunique()} ({len(dis_map)/df.disease.nunique():.0%})')
print('\nMapped diseases:', dis_map)
print('\nUnmapped symptoms:', [s for s in all_syms if s not in sym_map])

pd.DataFrame([(s, sym_map.get(s, '')) for s in all_syms], columns=['dataset_symptom', 'kg_symptom']).to_csv(f'{OUT}/symptom_mapping.csv', index=False)
pd.DataFrame([(d, dis_map.get(d, '')) for d in sorted(df.disease.unique())], columns=['dataset_disease', 'kg_disease']).to_csv(f'{OUT}/disease_mapping.csv', index=False)

## 5. KG-based components
* **Disease-support (A):** patient ke kitne symptoms KG mein har mapped disease se jude hain.
* **Symptom-expansion (B):** missing symptom ka score = present symptoms se KG similarity (shared diseases ki Jaccard).
* **KG-imputation:** B ko alag features banane ki bajaye, missing symptoms ki value directly fill karna (feature count same rehta hai → kam overfitting).
* **KG re-ranking:** model ki probabilities ko KG disease-support se adjust karna; weight λ validation par tune hota hai (λ=0 matlab KG ka koi asar nahi).

In [ ]:
kg_syms_used = sorted(set(sym_map.values()))
k_idx = {k: i for i, k in enumerate(kg_syms_used)}
dsets = [set(G.neighbors(k)) for k in kg_syms_used]
n = len(kg_syms_used)
SIM = np.zeros((n, n))
for a in range(n):
    for b in range(n):
        u = dsets[a] | dsets[b]
        SIM[a, b] = len(dsets[a] & dsets[b]) / len(u) if u else 0.0
np.fill_diagonal(SIM, 1.0)

mapped_dis = sorted(dis_map)
mapped_ds = sorted(sym_map)

def kg_parts(sym_lists):
    A = np.zeros((len(sym_lists), len(mapped_dis)))
    B = np.zeros((len(sym_lists), len(mapped_ds)))
    for r, ss in enumerate(sym_lists):
        ks = {sym_map[s] for s in ss if s in sym_map}
        if not ks:
            continue
        for c, d in enumerate(mapped_dis):
            A[r, c] = len(ks & kg_dis_syms[dis_map[d]]) / len(ks)
        present = [k_idx[k] for k in ks]
        for c, s in enumerate(mapped_ds):
            B[r, c] = SIM[present, k_idx[sym_map[s]]].max()
    return A, B

def kg_impute(sym_lists):
    X = onehot(sym_lists)
    _, B = kg_parts(sym_lists)
    for c, s in enumerate(mapped_ds):
        j = sym_idx[s]
        X[:, j] = np.maximum(X[:, j], B[:, c])
    return X

FEATS = {
    'onehot':     onehot,
    'kg_all':     lambda s: np.hstack([onehot(s), *kg_parts(s)]),
    'kg_support': lambda s: np.hstack([onehot(s), kg_parts(s)[0]]),
    'kg_impute':  kg_impute,
}

def rerank_predict(model, sym_lists, lam):
    P = model.predict_proba(onehot(sym_lists))
    logp = np.log(P + 1e-9)
    if lam and mapped_dis:
        A, _ = kg_parts(sym_lists)
        A = A - A.mean(axis=1, keepdims=True)
        cls = list(model.classes_)
        for c, d in enumerate(mapped_dis):
            if d in cls:
                logp[:, cls.index(d)] += lam * A[:, c]
    return model.classes_[logp.argmax(axis=1)]
print('Ready. Mapped diseases:', len(mapped_dis), '| mapped symptoms:', len(mapped_ds))

## 6. Experiment
6 variants × 2 models × 5 random splits × 4 dropout levels. **Aug** = training data mein bhi random symptom dropout (taake model incomplete data ke liye tayyar ho). Test set ka dropout har variant ke liye bilkul same hai (fair comparison).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

RATES = [0.0, 0.3, 0.5, 0.7]
SEEDS = range(5)
LAMBDAS = [0, 0.5, 1, 2, 4]
strat = df.disease if df.disease.value_counts().min() >= 2 else None

SETTINGS = {  # name: (features, augment, rerank)
    'Baseline':          ('onehot',     False, False),
    'Baseline+Aug':      ('onehot',     True,  False),
    'KG-features+Aug':   ('kg_all',     True,  False),
    'KG-support+Aug':    ('kg_support', True,  False),
    'KG-imputation+Aug': ('kg_impute',  True,  False),
    'KG-rerank+Aug':     ('onehot',     True,  True),
}
MODELS = {
    'LogReg':       lambda seed: LogisticRegression(max_iter=3000),
    'RandomForest': lambda seed: RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1),
}

def drop_symptoms(sym_lists, rate, rng):
    out = []
    for ss in sym_lists:
        ss = list(ss)
        keep = [s for s in ss if rng.random() > rate]
        out.append(tuple(keep) if keep else (ss[rng.integers(len(ss))],))
    return out

def augment(syms, labels, rng, copies=2, rates=(0.3, 0.5, 0.7)):
    S, Y = list(syms), list(labels)
    for r in rates:
        for _ in range(copies):
            S += drop_symptoms(syms, r, rng); Y += list(labels)
    return S, Y

def tune_lambda(make_model, tr, seed):
    s_in = tr.disease if tr.disease.value_counts().min() >= 2 else None
    itr, iva = train_test_split(tr, test_size=0.25, random_state=seed, stratify=s_in)
    rng = np.random.default_rng(seed + 999)
    S, Y = augment(list(itr.symptoms), list(itr.disease), rng)
    m = make_model().fit(onehot(S), Y)
    va = drop_symptoms(iva.symptoms, 0.5, rng)
    scores = {l: f1_score(iva.disease, rerank_predict(m, va, l), average='macro') for l in LAMBDAS}
    return max(scores, key=scores.get)

def fit_setting(sname, model, tr_syms, tr_y, aug_syms, aug_y):
    feat, aug, _ = SETTINGS[sname]
    S, Y = (aug_syms, aug_y) if aug else (tr_syms, tr_y)
    return model.fit(FEATS[feat](S), Y)

def predict_setting(sname, model, syms, lam):
    feat, _, rr = SETTINGS[sname]
    return rerank_predict(model, syms, lam) if rr else model.predict(FEATS[feat](syms))

rows, lams = [], []
for seed in SEEDS:
    tr, te = train_test_split(df, test_size=0.3, random_state=seed, stratify=strat)
    tr_syms, tr_y = list(tr.symptoms), list(tr.disease)
    aug_syms, aug_y = augment(tr_syms, tr_y, np.random.default_rng(seed + 500))
    test_sets = {r: drop_symptoms(te.symptoms, r, np.random.default_rng(seed * 100 + int(r * 10))) for r in RATES}
    for mname, mk in MODELS.items():
        lam = tune_lambda(lambda: mk(seed), tr, seed)
        lams.append(dict(seed=seed, model=mname, lam=lam))
        for sname in SETTINGS:
            model = fit_setting(sname, mk(seed), tr_syms, tr_y, aug_syms, aug_y)
            for r, ts in test_sets.items():
                pred = predict_setting(sname, model, ts, lam)
                rows.append(dict(seed=seed, model=mname, setting=sname, dropout=r,
                                 accuracy=accuracy_score(te.disease, pred),
                                 macro_f1=f1_score(te.disease, pred, average='macro')))
    print(f'seed {seed} done')

res = pd.DataFrame(rows)
lams = pd.DataFrame(lams)
print('Chosen λ for re-ranking:', lams.lam.tolist())

def table(model):
    g = res[res.model == model].groupby(['setting', 'dropout']).macro_f1.agg(['mean', 'std'])
    t = (g['mean'].map('{:.3f}'.format) + ' ± ' + g['std'].map('{:.3f}'.format)).unstack()
    t.columns = [f'{int(c*100)}% dropout' for c in t.columns]
    return t.loc[list(SETTINGS)]

for m in MODELS:
    print(f'\n=== {m} — macro-F1 (mean ± std over 5 splits) ===')
    display(table(m))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, m in zip(axes, MODELS):
    g = res[res.model == m]
    for sname in SETTINGS:
        s = g[g.setting == sname].groupby('dropout').macro_f1.mean()
        ax.plot(s.index, s.values, marker='o', label=sname,
                linestyle='--' if sname.startswith('Baseline') else '-')
    ax.set_title(m); ax.set_xlabel('Symptom dropout rate (test)'); ax.grid(alpha=.3)
axes[0].set_ylabel('Macro-F1'); axes[1].legend(fontsize=8)
plt.suptitle('Robustness to missing symptoms: baselines vs knowledge-graph variants')
plt.tight_layout(); plt.savefig(f'{OUT}/dropout_comparison.png', dpi=150); plt.show()

# Paired comparison vs Baseline+Aug at 50% dropout
p = res[res.dropout == 0.5].pivot_table(index=['seed', 'model'], columns='setting', values='macro_f1')
ref = 'Baseline+Aug'
comp = pd.DataFrame({s: {'mean_delta_vs_Baseline+Aug': (p[s] - p[ref]).mean(),
                         'wins': f"{int(((p[s] - p[ref]) > 0).sum())}/{len(p)}"}
                     for s in SETTINGS if s != ref}).T
comp['mean_delta_vs_Baseline+Aug'] = comp['mean_delta_vs_Baseline+Aug'].astype(float).round(3)
print('Paired comparison at 50% dropout (10 runs = 5 splits × 2 models):')
display(comp)

## 7. Explainability
Best KG variant ke predictions ko KG edges se trace karna.

In [ ]:
m50 = res[res.dropout == 0.5].groupby('setting').macro_f1.mean()
kg_names = [s for s in SETTINGS if s.startswith('KG')]
best_kg = m50[kg_names].idxmax()
best_model = res[(res.dropout == 0.5) & (res.setting == best_kg)].groupby('model').macro_f1.mean().idxmax()
print('Best KG variant:', best_kg, '|', best_model)

tr, te = train_test_split(df, test_size=0.3, random_state=0, stratify=strat)
tr_syms, tr_y = list(tr.symptoms), list(tr.disease)
aug_syms, aug_y = augment(tr_syms, tr_y, np.random.default_rng(500))
lam0 = lams[(lams.seed == 0) & (lams.model == best_model)].lam.iloc[0]
final = fit_setting(best_kg, MODELS[best_model](0), tr_syms, tr_y, aug_syms, aug_y)
te_syms = drop_symptoms(te.symptoms, 0.5, np.random.default_rng(42))
pred = predict_setting(best_kg, final, te_syms, lam0)

def explain(syms, disease):
    if disease not in dis_map:
        return None
    kd = dis_map[disease]
    return [f'{s} → {sym_map[s]}' for s in syms if s in sym_map and sym_map[s] in kg_dis_syms[kd]]

exp_rows = []
for syms, true, pr in zip(te_syms, te.disease, pred):
    ev = explain(syms, pr)
    exp_rows.append(dict(true=true, predicted=pr, symptoms=', '.join(syms), kg_disease=dis_map.get(pr),
                         n_evidence=len(ev) if ev is not None else np.nan, kg_evidence='; '.join(ev) if ev else ''))
exp = pd.DataFrame(exp_rows)
in_kg = exp.kg_disease.notna()
exp_cov_in_kg = in_kg.mean()
exp_cov_edge = (exp.loc[in_kg, 'n_evidence'] > 0).mean() if in_kg.any() else 0.0
print(f'Predictions whose disease is in KG: {exp_cov_in_kg:.0%}')
print(f'  ...of those, with ≥1 supporting KG edge: {exp_cov_edge:.0%}')
examples = exp[in_kg & (exp.n_evidence > 0) & (exp.true == exp.predicted)].head(3)
for _, r in examples.iterrows():
    print(f"\nPredicted: {r.predicted}\n  Symptoms: {r.symptoms}\n  KG evidence [{r.kg_disease}]: {r.kg_evidence}")

## 8. Data-quality layer
`kg_degree` = KG mein us disease ke total symptom edges. Kam degree wali diseases ke LOW flags zyada tar **KG ki kami** hain, data error nahi.

In [ ]:
dq = []
for _, r in df.iterrows():
    if r.disease not in dis_map:
        continue
    ks = {sym_map[s] for s in r.symptoms if s in sym_map}
    if len(ks) < 2:
        continue
    kd = dis_map[r.disease]
    support = len(ks & kg_dis_syms[kd]) / len(ks)
    dq.append(dict(disease=r.disease, kg_disease=kd, kg_degree=len(kg_dis_syms[kd]),
                   symptoms=', '.join(r.symptoms), kg_symptoms_checked=len(ks), kg_support=round(support, 2),
                   flag='INCONSISTENT' if support == 0 else ('LOW' if support < 0.34 else 'OK')))
dq = pd.DataFrame(dq)
if len(dq):
    print(dq.flag.value_counts())
    per_dis = dq.groupby(['disease', 'kg_degree']).kg_support.mean().round(2).sort_values()
    print('\nPer-disease KG support (with KG degree):'); print(per_dis)
    display(dq[dq.flag == 'INCONSISTENT'].head(15))

## 9. README + results save

In [ ]:
def md_table(t):
    lines = ['| Variant | ' + ' | '.join(t.columns) + ' |', '|' + '---|' * (len(t.columns) + 1)]
    lines += [f'| {i} | ' + ' | '.join(t.loc[i]) + ' |' for i in t.index]
    return '\n'.join(lines)

flags = dq.flag.value_counts().to_dict() if len(dq) else {}
d_ref, d_best = m50['Baseline+Aug'], m50[best_kg]
finding = (f"At 50% symptom dropout (macro-F1 averaged over both models), the best knowledge-graph variant "
           f"(**{best_kg}**) scored **{d_best:.3f}** vs **{d_ref:.3f}** for the augmented baseline "
           f"(Δ = {d_best - d_ref:+.3f}; better in {comp.loc[best_kg, 'wins']} paired runs). "
           f"Training-time symptom dropout alone changed the baseline from {m50['Baseline']:.3f} to {d_ref:.3f}. "
           f"Naively appending KG features to a model trained on complete data reduced robustness in an earlier run (v1).")

readme = f"""# Knowledge-Graph-Enhanced Disease Prediction

Hybrid pipeline combining an open biomedical knowledge graph (**Hetionet v1.0**, Disease–presents–Symptom edges) with machine-learning classifiers to predict diseases from symptoms, with a focus on **robustness to missing symptoms**, **explainability**, and **data quality**.

**Stack:** Python, pandas, scikit-learn, NetworkX, matplotlib (Google Colab)

## Data
- Symptom–disease dataset (Kaggle): {raw_rows} raw rows → **{len(df)} unique records** after de-duplication (prevents train/test leakage), {df.disease.nunique()} diseases, {len(all_syms)} symptoms.
- Knowledge graph: Hetionet — {len(kg_diseases)} diseases, {len(kg_symptoms)} symptoms (MeSH), {G.number_of_edges()} DpS edges.
- Mapping coverage (manual, see `symptom_mapping.csv`, `disease_mapping.csv`): **{len(sym_map)}/{len(all_syms)} symptoms**, **{len(dis_map)}/{df.disease.nunique()} diseases** (some disease matches are approximate).

## Method
Test-time symptom dropout (0–70%) simulates incomplete patient reports. 5 stratified 70/30 splits; identical dropped test sets across variants.

| Variant | Description |
|---|---|
| Baseline | One-hot symptoms, trained on complete records |
| Baseline+Aug | + training-time symptom dropout augmentation |
| KG-features+Aug | + KG disease-support and symptom-expansion features |
| KG-support+Aug | + KG disease-support features only |
| KG-imputation+Aug | Missing symptoms softly imputed from KG symptom similarity (shared diseases) |
| KG-rerank+Aug | Classifier probabilities re-ranked by KG disease support (weight tuned on validation) |

Models: Logistic Regression, Random Forest.

## Results (macro-F1, mean ± std over 5 splits)

### Logistic Regression
{md_table(table('LogReg'))}

### Random Forest
{md_table(table('RandomForest'))}

![dropout comparison](dropout_comparison.png)

**Key finding:** {finding}

## Explainability
Each prediction is traced to supporting Disease–presents–Symptom edges in the KG. On a 50%-dropout test set with {best_kg} ({best_model}): {exp_cov_in_kg:.0%} of predictions fall on KG-mapped diseases, and {exp_cov_edge:.0%} of those have ≥1 supporting KG edge (`explanations.csv`).

## Data-quality layer
Records are checked against KG knowledge: share of a record's KG-mapped symptoms linked to its labelled disease. Results: {flags}. Low support correlates with low KG degree of the disease, so many flags reflect **KG incompleteness** rather than data errors (`data_quality_flags.csv`).

## Limitations
- Hetionet covers only {len(dis_map)}/{df.disease.nunique()} dataset diseases; several mappings are approximate.
- The dataset is small and synthetic-like (near-perfect accuracy on complete records), so results are indicative only.
- Future work: larger KGs (e.g. PrimeKG), ontology-based mapping (UMLS/MeSH), real clinical data.

## How to run
Open `kg_disease_prediction_v2.ipynb` in Google Colab, upload the dataset zip to `/content`, Runtime → Run all.
"""
open(f'{OUT}/README.md', 'w').write(readme)
open(f'{OUT}/requirements.txt', 'w').write('pandas\nnumpy\nscikit-learn\nnetworkx\nmatplotlib\n')
res.to_csv(f'{OUT}/results_all_runs.csv', index=False)
lams.to_csv(f'{OUT}/rerank_lambdas.csv', index=False)
comp.to_csv(f'{OUT}/paired_comparison_50pct.csv')
exp.to_csv(f'{OUT}/explanations.csv', index=False)
dq.to_csv(f'{OUT}/data_quality_flags.csv', index=False)
for m in MODELS:
    table(m).to_csv(f'{OUT}/results_{m}.csv')

import shutil
shutil.make_archive(f'{BASE}/results', 'zip', OUT)
print(finding)
try:
    from google.colab import files
    files.download(f'{BASE}/results.zip')
except ImportError:
    print('Saved to', OUT)